# Analisis Sentimen Pilpres 2024: Perbandingan RNN, LSTM, dan GRU

| Nama                      | NPM        |
| ------------------------- | ---------- |
| Muhammad Hilmi Al Muttaqi | 2306267082 |
| Reyhan Ahnaf Deannova     | 2306267100 |
| Bryan Herdianto           | 2306210885 |
| Adhikananda Wira Januar   | 2306267113 |

Notebook ini membandingkan tiga arsitektur Recurrent Neural Network (**RNN**, **LSTM**, dan **GRU**) untuk mengklasifikasikan sentimen opini publik terkait Pemilihan Presiden Indonesia 2024.

Secara umum, alur kerjanya:
1. **Load Dataset** — mengunduh data tweet berlabel sentimen dari Kaggle
2. **Preprocessing** — membersihkan teks dari URL, mention, tanda baca, dll.
3. **Tokenisasi & Encoding** — mengubah teks menjadi angka agar bisa diproses model
4. **Bangun Model** — membuat tiga model (RNN, LSTM, GRU) dengan arsitektur yang setara
5. **Training** — melatih ketiga model dan menyimpan metriknya
6. **Evaluasi** — membandingkan akurasi, precision, recall, F1-score, dan waktu training
7. **Visualisasi** — menampilkan grafik loss/accuracy dan confusion matrix

In [ ]:
import os
import re
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight

import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, LSTM, GRU, Dense, Dropout, Bidirectional
from tensorflow.keras.callbacks import EarlyStopping

import kagglehub

## 1. Load Dataset

Dataset diambil dari Kaggle dan berisi tweet yang sudah diberi label sentimen (**Positive**, **Negative**, **Neutral**) untuk tiga pasangan calon presiden: Anies Baswedan, Ganjar Pranowo, dan Prabowo Subianto.

Kita akan mengunduh dataset, mencari file CSV-nya secara otomatis, lalu menggabungkan ketiga file menjadi satu dataframe.

In [ ]:
print("Mengunduh dataset dari Kaggle...")
dataset_path = kagglehub.dataset_download("jocelyndumlao/indonesia-presidential-candidates-dataset-2024")
print(f"Dataset root folder: {dataset_path}")

path_anies = None
path_ganjar = None
path_prabowo = None

for root, dirs, files in os.walk(dataset_path):
    if 'labeled' in root.lower():
        for file in files:
            if "Anies Baswedan" in file and file.endswith('.csv'):
                path_anies = os.path.join(root, file)
            elif "Ganjar Pranowo" in file and file.endswith('.csv'):
                path_ganjar = os.path.join(root, file)
            elif "Prabowo Subianto" in file and file.endswith('.csv'):
                path_prabowo = os.path.join(root, file)

if not all([path_anies, path_ganjar, path_prabowo]):
    raise FileNotFoundError("Salah satu atau semua file CSV tidak ditemukan!")

print("\nBerhasil menemukan file:")
print(f"- {path_anies}")
print(f"- {path_ganjar}")
print(f"- {path_prabowo}")

In [ ]:
df_anies = pd.read_csv(path_anies)
df_ganjar = pd.read_csv(path_ganjar)
df_prabowo = pd.read_csv(path_prabowo)
df = pd.concat([df_anies, df_ganjar, df_prabowo], ignore_index=True)
df = df.dropna(subset=['Text', 'label'])

print(f"Total data bersih siap diproses: {df.shape[0]} baris")
print("\nDistribusi kelas:")
print(df['label'].value_counts())

## 2. Text Preprocessing

Sebelum teks bisa diproses oleh model, kita perlu membersihkannya terlebih dahulu. Langkah-langkah pembersihan yang dilakukan:
- **Case folding** — mengubah semua huruf menjadi huruf kecil
- **Hapus URL** — menghilangkan link yang ada di tweet
- **Hapus mention dan hashtag** — menghilangkan `@username` dan `#tag`
- **Hapus tanda baca dan angka** — hanya menyisakan huruf dan spasi
- **Strip** — menghapus spasi berlebih di awal dan akhir teks

Teks yang bersih akan membantu model belajar pola sentimen dengan lebih baik.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    text = re.sub(r'\@\w+|\#', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\d+', '', text)
    text = text.strip()
    return text

df['cleaned_text'] = df['Text'].apply(clean_text)
print("Text preprocessing selesai.")
print(f"Contoh teks asli : {df['Text'].iloc[0]}")
print(f"Contoh teks bersih: {df['cleaned_text'].iloc[0]}")

## 3. Tokenisasi & Label Encoding

Model tidak bisa memproses teks mentah, jadi kita perlu mengubahnya menjadi angka:
- **Tokenisasi** — setiap kata diubah menjadi indeks angka berdasarkan kamus yang dibangun dari data. Kita membatasi kamusnya sebanyak 10.000 kata (`max_words`) dan panjang kalimat maksimal 100 kata (`max_len`).
- **Padding** — menyamakan panjang semua kalimat agar seragam (kalimat pendek ditambah nol, kalimat panjang dipotong).
- **Label Encoding** — mengubah label sentimen (Positive/Negative/Neutral) menjadi angka (0/1/2).
- **Class Weight** — karena jumlah data per kelas tidak seimbang, kita hitung bobot kelas agar model tidak cenderung memihak kelas mayoritas.

Data dibagi menjadi **80% training** dan **20% testing**.

In [ ]:
le = LabelEncoder()
y = le.fit_transform(df['label'])
num_classes = len(np.unique(y))
target_names = le.classes_

print(f"Label classes: {list(enumerate(target_names))}")
print(f"Jumlah kelas: {num_classes}")

max_words = 10000
max_len = 100

tokenizer = Tokenizer(num_words=max_words, oov_token="<OOV>")
tokenizer.fit_on_texts(df['cleaned_text'])

X = tokenizer.texts_to_sequences(df['cleaned_text'])
X = pad_sequences(X, maxlen=max_len, padding='post', truncating='post')

print(f"Vocabulary size: {len(tokenizer.word_index)}")
print(f"Shape X: {X.shape}, Shape y: {y.shape}")

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=10, stratify=y)

print(f"Training set: {X_train.shape[0]} sampel")
print(f"Testing set : {X_test.shape[0]} sampel")

class_weights_arr = compute_class_weight(class_weight='balanced', classes=np.unique(y_train), y=y_train)
class_weight = dict(enumerate(class_weights_arr))
print("\nClass weights (untuk mengatasi imbalance):")
for k, v in class_weight.items():
    print(f"  Kelas {k} ({target_names[k]}): {v:.4f}")

## 4. Bangun Model

Kita membangun tiga model dengan arsitektur yang setara agar perbandingannya adil:

| Komponen | Keterangan |
|---|---|
| **Embedding** | Mengubah kata (indeks angka) menjadi vektor 128 dimensi |
| **Recurrent Layer** | Simple RNN / Bidirectional LSTM / Bidirectional GRU dengan 128 unit |
| **Dense** | Layer fully-connected 64 neuron dengan aktivasi ReLU + Dropout 0.3 |
| **Output** | Softmax dengan jumlah neuron = jumlah kelas sentimen |

> **Kenapa Bidirectional untuk LSTM dan GRU?** Bidirectional membaca teks dari kiri-ke-kanan **dan** kanan-ke-kiri, sehingga model bisa menangkap konteks dari kedua arah. Simple RNN tetap menggunakan arsitektur satu arah sebagai baseline.

> **Kenapa pakai class_weight?** Karena jumlah data per kelas tidak seimbang, class_weight memberikan bobot lebih besar pada kelas yang sedikit datanya agar model tidak hanya memprediksi kelas mayoritas.

In [ ]:
def build_model(model_type):
    model = Sequential()
    model.add(Embedding(input_dim=max_words, output_dim=128))

    if model_type == 'Simple RNN':
        model.add(SimpleRNN(128, dropout=0.3, recurrent_dropout=0.2))
    elif model_type == 'LSTM':
        model.add(Bidirectional(LSTM(128, dropout=0.3, recurrent_dropout=0.2)))
    elif model_type == 'GRU':
        model.add(Bidirectional(GRU(128, dropout=0.3, recurrent_dropout=0.2)))

    model.add(Dense(64, activation='relu'))
    model.add(Dropout(0.3))
    model.add(Dense(num_classes, activation='softmax'))

    model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
    return model

## 5. Training Model

Ketiga model dilatih dengan konfigurasi yang sama:
- **Epoch** = 30 (maksimal), dengan **EarlyStopping** patience=5 (berhenti jika val_loss tidak membaik selama 5 epoch)
- **Batch size** = 64
- **Validation split** = 10% dari data training
- **Class weight** diterapkan agar model memperhatikan kelas minoritas

Setelah training, model diuji pada data testing dan metrik berikut dihitung: Accuracy, Precision, Recall, F1-Score.

In [ ]:
models_to_train = ['Simple RNN', 'LSTM', 'GRU']
results = {}
histories = {}

epochs = 30
batch_size = 64
early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)

for model_name in models_to_train:
    print(f"\nTraining Model: {model_name}")
    print("-" * 40)

    model = build_model(model_name)
    start_time = time.time()

    history = model.fit(
        X_train, y_train,
        epochs=epochs,
        batch_size=batch_size,
        validation_split=0.1,
        class_weight=class_weight,
        callbacks=[early_stop],
        verbose=1
    )

    training_time = time.time() - start_time
    histories[model_name] = history

    y_pred_prob = model.predict(X_test)
    y_pred = np.argmax(y_pred_prob, axis=1)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, average='weighted', zero_division=0)
    rec = recall_score(y_test, y_pred, average='weighted', zero_division=0)
    f1 = f1_score(y_test, y_pred, average='weighted', zero_division=0)
    cm = confusion_matrix(y_test, y_pred)

    results[model_name] = {
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1-Score': f1,
        'Training Time (s)': training_time,
        'Confusion Matrix': cm
    }

    print(f"Selesai! Accuracy: {acc:.4f}, F1-Score: {f1:.4f}, Waktu: {training_time:.1f}s")

## 6. Tabel Perbandingan Performa

Berikut adalah perbandingan metrik kinerja ketiga model. Semakin tinggi nilai Accuracy, Precision, Recall, dan F1-Score, semakin baik performa model tersebut.

In [ ]:
print("PERBANDINGAN KINERJA MODEL")
print("=" * 60)
results_df = pd.DataFrame(results).T
display_df = results_df[['Accuracy', 'Precision', 'Recall', 'F1-Score', 'Training Time (s)']]
print(display_df.round(4).to_string())

## 7. Visualisasi Hasil Training

### Grafik Akurasi & Loss per Epoch

Grafik di bawah menunjukkan perbandingan **akurasi validasi** dan **loss validasi** ketiga model di setiap epoch. Model yang baik memiliki akurasi yang naik dan loss yang turun secara konsisten.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for model_name, history in histories.items():
    axes[0].plot(history.history['val_accuracy'], label=f'{model_name}')
    axes[1].plot(history.history['val_loss'], label=f'{model_name}')

axes[0].set_title('Perbandingan Akurasi Validasi')
axes[0].set_xlabel('Epochs')
axes[0].set_ylabel('Accuracy')
axes[0].legend()
axes[0].grid(True, linestyle='--', alpha=0.6)

axes[1].set_title('Perbandingan Loss Validasi')
axes[1].set_xlabel('Epochs')
axes[1].set_ylabel('Loss')
axes[1].legend()
axes[1].grid(True, linestyle='--', alpha=0.6)

plt.tight_layout()
plt.show()

### Confusion Matrix

Confusion matrix menunjukkan distribusi prediksi model terhadap label aktual. Diagonal utama menunjukkan jumlah prediksi yang benar, sedangkan sel lainnya menunjukkan kesalahan prediksi.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, model_name in enumerate(models_to_train):
    sns.heatmap(results[model_name]['Confusion Matrix'],
                annot=True, fmt='d', cmap='Blues',
                ax=axes[i], xticklabels=target_names, yticklabels=target_names)
    axes[i].set_title(f'Confusion Matrix: {model_name}')
    axes[i].set_xlabel('Label Prediksi')
    axes[i].set_ylabel('Label Aktual')

plt.tight_layout()
plt.show()

## Ringkasan

Cell di bawah ini akan menampilkan ringkasan distribusi kelas dan metrik performa model jika cell-cell sebelumnya sudah dijalankan.

In [ ]:
try:
    print('Distribusi Kelas:')
    print(df['label'].value_counts())
    print('\nRingkasan Metrik:')
    display(display_df)
except NameError as e:
    print(f"Error: {e}")
    print("Jalankan semua cell di atas terlebih dahulu.")